In [207]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## **1. Profiling**

---
### **Checking for Null Values and Data Types**

Note: "Rules Satisfied" refers to the consistency with the data description given from the instructions, if Rules Satisfied = n(rows) then there are no violations based on the data description.

In [208]:
import os

import pandas as pd
import numpy as np


def null_report(df):
    """Report null/NA values per column of a single table.

    For every column returns:
      - column name
      - number of null or NA values
      - percentage of those null values relative to the table's row count
      - number of duplicate values in the column

    Parameters
    ----------
    df : pd.DataFrame
        A single table.

    Returns
    -------
    pd.DataFrame
        One row per column with the columns:
        ['column', 'null_count', 'null_percentage', 'duplicate_count'].
    """
    total_rows = len(df)

    report = pd.DataFrame({
        "column": df.columns,
        "null_count": df.isna().sum().values,
        "duplicate_count": [int(df[col].duplicated().sum()) for col in df.columns],
    })

    report["null_percentage"] = np.where(
        total_rows > 0,
        report["null_count"] / total_rows * 100,
        0.0,
    ).round(2)

    return report.reset_index(drop=True)


# Expected values per table, derived from table_info.md (row counts ignored).
# Each entry maps a column name to a predicate the value must satisfy, or None
# when no specific rule is defined. Used by the completeness/consistency check.
TABLE_EXPECTATIONS = {
    "olist_orders_dataset": {
        "order_id": "not_null",
        "customer_id": "not_null",
        "order_status": lambda s: s.isin(
            ["created", "approved", "invoiced", "processing",
             "shipped", "delivered", "unavailable", "canceled"]
        ),
        "order_purchase_timestamp": "not_null",
        "order_approved_at": None,
        "order_delivered_carrier_date": None,
        "order_delivered_customer_date": None,
        "order_estimated_delivery_date": "not_null",
    },
    "olist_order_items_dataset": {
        "order_id": "not_null",
        "order_item_id": "not_null",
        "product_id": "not_null",
        "seller_id": "not_null",
        "shipping_limit_date": "not_null",
        "price": lambda s: pd.to_numeric(s, errors="coerce").ge(0),
        "freight_value": lambda s: pd.to_numeric(s, errors="coerce").ge(0),
    },
    "olist_order_payments_dataset": {
        "order_id": "not_null",
        "payment_sequential": "not_null",
        "payment_type": lambda s: s.isin(
            ["credit_card", "boleto", "voucher", "debit_card", "not_defined"]
        ),
        "payment_installments": lambda s: pd.to_numeric(s, errors="coerce").ge(0),
        "payment_value": lambda s: pd.to_numeric(s, errors="coerce").ge(0),
    },
    "olist_order_reviews_dataset": {
        "review_id": "not_null",
        "order_id": "not_null",
        "review_score": lambda s: pd.to_numeric(s, errors="coerce").between(1, 5),
        "review_comment_title": None,
        "review_comment_message": None,
        "review_creation_date": "not_null",
        "review_answer_timestamp": "not_null",
    },
    "olist_customers_dataset": {
        "customer_id": "not_null",
        "customer_unique_id": "not_null",
        "customer_zip_code_prefix": "not_null",
        "customer_city": "not_null",
        "customer_state": "not_null",
    },
    "olist_products_dataset": {
        "product_id": "not_null",
        "product_category_name": None,
        "product_name_lenght": None,
        "product_description_lenght": None,
        "product_photos_qty": None,
        "product_weight_g": None,
        "product_length_cm": None,
        "product_height_cm": None,
        "product_width_cm": None,
    },
    "olist_sellers_dataset": {
        "seller_id": "not_null",
        "seller_zip_code_prefix": "not_null",
        "seller_city": "not_null",
        "seller_state": "not_null",
    },
    "olist_geolocation_dataset": {
        "geolocation_zip_code_prefix": "not_null",
        "geolocation_lat": "not_null",
        "geolocation_lng": "not_null",
        "geolocation_city": "not_null",
        "geolocation_state": "not_null",
    },
    "product_category_name_translation": {
        "product_category_name": "not_null",
        "product_category_name_english": "not_null",
    },
}


def _satisfies(series, rule):
    """Return a boolean Series telling whether each value meets ``rule``."""
    if rule is None:
        return pd.Series(True, index=series.index)
    if rule == "not_null":
        return series.notna()
    return rule(series).fillna(False)


def check_table(df, table_name, expectations=None):
    """Report the completeness and consistency of the columns of one table.

    Parameters
    ----------
    df : pd.DataFrame
        A single table.
    table_name : str
        Name of the table (key used to look up expectations).
    expectations : dict, optional
        Mapping of column name -> rule. Defaults to ``TABLE_EXPECTATIONS``.
        A rule is either ``"not_null"``, a callable returning a boolean Series,
        or ``None`` (no specific rule).

    Returns
    -------
    pd.DataFrame
        One row per column with:
        ['column', 'total', 'non_null', 'completeness_pct',
         'rule_satisfied', 'violations', 'consistency_pct', 'consistent'].
    """
    if expectations is None:
        expectations = TABLE_EXPECTATIONS
    rules = expectations.get(table_name, {})

    total = len(df)
    rows = []
    for col in df.columns:
        series = df[col]
        non_null = int(series.notna().sum())
        completeness = round(non_null / total * 100, 2) if total else 0.0

        rule = rules.get(col, None)
        if rule is None:
            satisfied = pd.Series(True, index=series.index)
        else:
            satisfied = _satisfies(series, rule) & series.notna()

        violations = int((~satisfied).sum())
        consistency = round(satisfied.sum() / total * 100, 2) if total else 0.0

        rows.append({
            "column": col,
            "total": total,
            "non_null": non_null,
            "completeness_pct": completeness,
            "rule_satisfied": int(satisfied.sum()),
            "violations": violations,
            "consistency_pct": consistency,
            "consistent": violations == 0,
        })

    return pd.DataFrame(rows)


def check_all_tables(folder=".", pattern="*.csv", expectations=None):
    """Run ``check_table`` on every .csv in a folder.

    Parameters
    ----------
    folder : str, optional
        Folder to scan. Defaults to the current directory (the notebook's folder).
    pattern : str, optional
        Glob pattern for the files to load (default ``"*.csv"``).
    expectations : dict, optional
        Passed through to ``check_table``.

    Returns
    -------
    dict[str, pd.DataFrame]
        Mapping of table name (file name without extension) -> report.
    """
    reports = {}
    for path in sorted(glob.glob(os.path.join(folder, pattern))):
        table_name = os.path.splitext(os.path.basename(path))[0]
        reports[table_name] = check_table(
            pd.read_csv(path), table_name, expectations=expectations
        )
    return reports


def table_datatypes(tables):
    """Return a dataframe with the datatypes of every column of every table.

    Parameters
    ----------
    tables : dict[str, pd.DataFrame]
        Mapping of table name -> dataframe.

    Returns
    -------
    pd.DataFrame
        One row per (table, column) with ['table', 'column', 'dtype'].
    """
    rows = []
    for name, df in tables.items():
        for col in df.columns:
            rows.append({
                "table": name,
                "column": col,
                "dtype": str(df[col].dtype),
            })
    return pd.DataFrame(rows)

In [209]:
# Null report for all tables 
print('--- per table check ---')
for name, report in check_all_tables().items():
    print(f'=== {name} ===')
    print(report.to_string())
    print('---')

--- per table check ---
=== olist_customers_dataset ===
                     column  total  non_null  completeness_pct  rule_satisfied  violations  consistency_pct  consistent
0               customer_id  99441     99441             100.0           99441           0            100.0        True
1        customer_unique_id  99441     99441             100.0           99441           0            100.0        True
2  customer_zip_code_prefix  99441     99441             100.0           99441           0            100.0        True
3             customer_city  99441     99441             100.0           99441           0            100.0        True
4            customer_state  99441     99441             100.0           99441           0            100.0        True
---
=== olist_geolocation_dataset ===
                        column    total  non_null  completeness_pct  rule_satisfied  violations  consistency_pct  consistent
0  geolocation_zip_code_prefix  1000163   1000163            

In [210]:
tables = {
    'orders': pd.read_csv('olist_orders_dataset.csv', nrows=5),
    'order_items': pd.read_csv('olist_order_items_dataset.csv', nrows=5),
    'order_payments': pd.read_csv('olist_order_payments_dataset.csv', nrows=5),
    'reviews': pd.read_csv('olist_order_reviews_dataset.csv', nrows=5),
    'customers': pd.read_csv('olist_customers_dataset.csv', nrows=5),
    'products': pd.read_csv('olist_products_dataset.csv', nrows=5),
    'sellers': pd.read_csv('olist_sellers_dataset.csv', nrows=5),
    'geolocation': pd.read_csv('olist_geolocation_dataset.csv', nrows=5),
    'product_category_name_translation': pd.read_csv('product_category_name_translation.csv', nrows=5)
}
print('--- multi-table datatypes ---')
print(table_datatypes(tables).to_string())

--- multi-table datatypes ---
                                table                         column    dtype
0                              orders                       order_id      str
1                              orders                    customer_id      str
2                              orders                   order_status      str
3                              orders       order_purchase_timestamp      str
4                              orders              order_approved_at      str
5                              orders   order_delivered_carrier_date      str
6                              orders  order_delivered_customer_date      str
7                              orders  order_estimated_delivery_date      str
8                         order_items                       order_id      str
9                         order_items                  order_item_id    int64
10                        order_items                     product_id      str
11                        order_it

---
### **Per Table Checks**

In [211]:
orders_df = pd.read_csv('olist_orders_dataset.csv')
order_items_df = pd.read_csv('olist_order_items_dataset.csv') #fact table
order_payments_df = pd.read_csv('olist_order_payments_dataset.csv') #fact table
order_reviews_df = pd.read_csv('olist_order_reviews_dataset.csv') #fact table
customers_df = pd.read_csv('olist_customers_dataset.csv')
products_df = pd.read_csv('olist_products_dataset.csv')
sellers_df = pd.read_csv('olist_sellers_dataset.csv')
geolocation_df = pd.read_csv('olist_geolocation_dataset.csv')
product_category_name_translation_df = pd.read_csv('product_category_name_translation.csv')

#### **Orders**

In [212]:
orders_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [213]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [214]:
#check if all occurences of customer_id also occur in the customers_df; referential integrity
orders_df['customer_id'].isin(customers_df['customer_id']).value_counts()

customer_id
True    99441
Name: count, dtype: int64

In [215]:
orders_df['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [216]:
orders_df[orders_df['order_approved_at'].isnull()].T

,1130,1801,1868,2029,2161,3056,3094,3684,4373,4939,...,93992,94252,95451,96996,97100,97696,98415,98909,99283,99347
order_id,00b1cb0320190ca0daa2c88b35206009,ed3efbd3a87bea76c2812c66a0b32219,df8282afe61008dc26c6c31011474d02,8d4c637f1accf7a88a4555f02741e606,7a9d4c7f9b068337875b95465330f2fc,ddaec6fff982b13e7e048b627a11d6da,5290c34bd38a8a095b885f13958db1e1,03310aa823a66056268a3bab36e827fb,4c8b9947280829d0a8b7e81cc249b875,b13ea375fe9c728832688264638f84cf,...,b6e63e74aff84a068fdc1b41e9ac29ea,72fcdcc4d106595993b31fc91c800113,7813842ae95e8c497fc0233232ae815a,5a14c8b3d919a4ef3f3428b0459c47b2,b34369ee8397436dd7ceb8bb7783671d,5a00b4d35edffc56b825c3646a99ba9d,227c804e2a44760671a6a5697ea549e4,e49e7ce1471b4693482d40c2bd3ad196,3a3cddda5a7c27851bd96c3313412840,a89abace0dcc01eeb267a9660b5ac126
customer_id,3532ba38a3fd242259a514ac2b6ae6b6,191984a8ba4cbb2145acb4fe35b69664,aa797b187b5466bc6925aaaa4bb3bed1,b1dd715db389a2077f43174e7a675d07,7f71ae48074c0cfec9195f88fcbfac55,68f4ad79cc0c2ad06e19088f5c00e9fa,92af427e290117f39d9ff908566072e0,25dbbf0c477fd4ae0880aaffbb12e8b3,403c35c4d8813bf67b3d396b91ca1619,0dc5884bc5ffba283678229f27e07ff9,...,f0375e81392cb7499b1139be16fd21e2,89cbb4528dbfd3ec138d5014dc8ccb0e,040d94f8ba8ca26014bd6f7e8a6e0c0d,666094835d60d986eb87350b31efdcae,c96771504b0c9d89ad8da161e0e3d092,6a3bdf004ca96338fb5fad1b8d93c2e6,62e7477e75e542243ee62a0ba73f410f,e4e7ab3f449aeb401f0216f86c2104db,0b0d6095c5555fe083844281f6b093bb,2f0524a7b1b3845a1a57fcf3910c4333
order_status,canceled,canceled,canceled,canceled,canceled,canceled,canceled,canceled,canceled,canceled,...,canceled,canceled,canceled,canceled,canceled,canceled,canceled,canceled,canceled,canceled
order_purchase_timestamp,2018-08-28 15:26:39,2018-09-20 13:54:16,2017-03-04 12:14:30,2018-08-29 16:27:49,2017-05-01 16:12:39,2016-10-04 19:41:32,2018-08-21 10:25:18,2018-08-07 16:33:59,2018-08-09 14:54:47,2018-08-22 18:52:29,...,2018-08-07 10:44:34,2017-09-12 22:36:04,2018-08-17 20:06:36,2017-05-29 23:53:39,2017-06-07 20:11:29,2017-07-02 15:38:46,2017-09-28 15:02:56,2018-08-07 11:16:28,2018-08-31 16:13:44,2018-09-06 18:45:47
order_approved_at,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_delivered_carrier_date,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_delivered_customer_date,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_estimated_delivery_date,2018-09-12 00:00:00,2018-10-17 00:00:00,2017-04-10 00:00:00,2018-09-13 00:00:00,2017-05-30 00:00:00,2016-11-16 00:00:00,2018-09-06 00:00:00,2018-09-04 00:00:00,2018-08-21 00:00:00,2018-09-19 00:00:00,...,2018-08-21 00:00:00,2017-09-22 00:00:00,2018-09-17 00:00:00,2017-06-23 00:00:00,2017-07-04 00:00:00,2017-07-25 00:00:00,2017-10-16 00:00:00,2018-08-10 00:00:00,2018-10-01 00:00:00,2018-09-27 00:00:00


In [217]:
orders_df[orders_df['order_approved_at'].isnull()]['order_status'].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

Most of the instances where order_approved_at was null were cancellled orders

In [218]:
orders_df[orders_df['order_delivered_carrier_date'].isnull()]['order_status'].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

The same applies here except for the two instances where the status was 'delivered'

In [219]:
orders_df[orders_df['order_delivered_customer_date'].isnull()]['order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

#### **Order_Item**

In [220]:
order_items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


shipping_limit_date is a string

### **Zip-prefixes**

In [221]:
print("Postal-code length for all entries in customers is 5: ", len(customers_df[customers_df['customer_zip_code_prefix'] == 5]) == len(customers_df['customer_zip_code_prefix']))
print("Postal-code length for all entries in sellers is 5: ", len(sellers_df[sellers_df['seller_zip_code_prefix'] == 5]) == len(sellers_df['seller_zip_code_prefix']))
print("Postal-code length for all entries in geolocation is 5: ", len(geolocation_df[geolocation_df['geolocation_zip_code_prefix'] == 5]) == len(geolocation_df['geolocation_zip_code_prefix']))

Postal-code length for all entries in customers is 5:  False
Postal-code length for all entries in sellers is 5:  False
Postal-code length for all entries in geolocation is 5:  False


In [222]:
customers_df[customers_df['customer_zip_code_prefix'].astype(str).str.len() != 5]['customer_zip_code_prefix']

1        9790
2        1151
3        8775
6        4534
13       5704
         ... 
99430    6727
99434    7790
99436    3937
99437    6764
99440    6703
Name: customer_zip_code_prefix, Length: 23995, dtype: int64

In [223]:
#where the first character of the postal code is zero
customers_df[customers_df['customer_zip_code_prefix'].astype(str).str[0] == '0']

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state


In [224]:
print(len(customers_df[customers_df['customer_zip_code_prefix'].astype(str).str.len() != 5]) == len(customers_df[customers_df['customer_zip_code_prefix'].astype(str).str.len() == 4]))
print(len(sellers_df[sellers_df['seller_zip_code_prefix'].astype(str).str.len() != 5]) == len(sellers_df[sellers_df['seller_zip_code_prefix'].astype(str).str.len() == 4]))
print(len(geolocation_df[geolocation_df['geolocation_zip_code_prefix'].astype(str).str.len() != 5]) == len(geolocation_df[geolocation_df['geolocation_zip_code_prefix'].astype(str).str.len() == 4]))

True
True
True


In [225]:
print(customers_df['customer_zip_code_prefix'].dtype)
print(sellers_df['seller_zip_code_prefix'].dtype)
print(geolocation_df['geolocation_zip_code_prefix'].dtype)

int64
int64
int64


The reason for the loss of the leading zero is because the zip codes were read as integers. The previous cells also show that the number of postal code entries for the three postal code columns where the postal code length is equal to 4 is the same as the number of postal code entries that are NOT equal to 5. Hence, the reason for this discrepancy is the lack of leading zeroes. 

### **Check if values are within range**

In [226]:
#check if order_status in created, approved, invoiced, processing, shipped, delivered, unavailable, canceled
orders_df['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [227]:
#check if payment_type in credit_card, boleto, voucher, debit_card, not_defined
order_payments_df['payment_type'].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [228]:
orders_df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

In [229]:
#check if all instances of price, freight_value, payment_value, payment_installments are >= 0
print(order_payments_df[order_payments_df['payment_value'] < 0])
print(order_payments_df[order_payments_df['payment_installments'] < 0])

print(order_items_df[order_items_df['price'] < 0])
print(order_items_df[order_items_df['freight_value'] < 0])

Empty DataFrame
Columns: [order_id, payment_sequential, payment_type, payment_installments, payment_value]
Index: []
Empty DataFrame
Columns: [order_id, payment_sequential, payment_type, payment_installments, payment_value]
Index: []
Empty DataFrame
Columns: [order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value]
Index: []
Empty DataFrame
Columns: [order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value]
Index: []


In [230]:
#check if review score is in the range 1-5
print(True if order_reviews_df['review_score'].between(1, 5).all() else False)

True


In [231]:
#check for null values for id columns 
def check_id_uniqueness(df: pd.DataFrame) -> pd.DataFrame:
    """Check uniqueness of every column whose name contains 'ID'.

    Columns are matched if their name contains 'ID' (case-insensitive). A
    column is considered unique when it has no duplicate non-null values.
    'n duplicates' counts the number of extra occurrences beyond the first,
    i.e. total rows minus the number of distinct non-null values.

    Returns a DataFrame with columns: "column name", "unique", "n duplicates".
    """
    id_cols = [col for col in df.columns if "id" in str(col).lower()]

    results = []
    for col in id_cols:
        non_null = df[col].dropna()
        n_duplicates = int(non_null.duplicated().sum())
        results.append(
            {
                "column name": col,
                "unique": n_duplicates == 0,
                "n duplicates": n_duplicates,
            }
        )

    return pd.DataFrame(
        results, columns=["column name", "unique", "n duplicates"]
    )


df_list = [orders_df,
order_items_df,
order_payments_df,
order_reviews_df,
customers_df, 
products_df,
sellers_df,
geolocation_df,
product_category_name_translation_df]

for df in df_list:
    print(f"--- {df.columns[0]} ---")
    print(check_id_uniqueness(df).to_string())
    print()

--- order_id ---
   column name  unique  n duplicates
0     order_id    True             0
1  customer_id    True             0

--- order_id ---
     column name  unique  n duplicates
0       order_id   False         13984
1  order_item_id   False        112629
2     product_id   False         79699
3      seller_id   False        109555

--- order_id ---
  column name  unique  n duplicates
0    order_id   False          4446

--- review_id ---
  column name  unique  n duplicates
0   review_id   False           814
1    order_id   False           551

--- customer_id ---
          column name  unique  n duplicates
0         customer_id    True             0
1  customer_unique_id   False          3345

--- product_id ---
        column name  unique  n duplicates
0        product_id    True             0
1  product_width_cm   False         32854

--- seller_id ---
  column name  unique  n duplicates
0   seller_id    True             0

--- geolocation_zip_code_prefix ---
Empty DataFrame

### **Findings from Profiling**

**1. Timestamps are strings**

Columns that are expected to be a date is read as an object or string (order_\*, shipping_limit_date, review_\*) 


**2. Zip-code prefixes lacking leading zeros**

Normalize all zip prefixes to add zeroes for the zip-code entries to have a character length of 5. 

**3. Null Values**

Only orders, products and the reviews table have null values. 

Non-Problematic Null Values: Some of the timestamps for the orders table are null because the order either never or was yet to be received (cancelled, unavailable or still processing). These null values are not to be dropped or imputed. The reviews table contains multiple null values, but these are simply instances where the customer left no comment, hence those rows may be expected to be null. 

Problematic Null Values: The products table has 610 missing rows in the category name, name length, and photos quantity column. There are 2 missing rows for product weight, length, height and width. 

**4. Expected Range**

Order status must be only either ‘created’, ‘approved’, ‘invoiced’, ‘processing’, ‘shipped’, ‘delivered’, ‘unavailable’, or ‘canceled’
Payment type must be only either ‘credit_card’, ‘boleto’, ‘voucher’, ‘debit_card’, or ‘not_defined’
Price, freight_value, payment_value, payment_installments must strictly be greater than zero. 
Review score must be in the range [1,5]


In [232]:
#optional to run, files must be in the same folder, uncomment to run
#%run 00_stage.py  
#%run 01_profile.py

## **2. Cleaning**

### Load Staged Data

In [233]:
RAW = "staging"          # folder with the unmodified files (created in step 1)

orders      = pd.read_csv(f"{RAW}/olist_orders_dataset.csv")
order_items = pd.read_csv(f"{RAW}/olist_order_items_dataset.csv")
payments    = pd.read_csv(f"{RAW}/olist_order_payments_dataset.csv")
reviews     = pd.read_csv(f"{RAW}/olist_order_reviews_dataset.csv")
customers   = pd.read_csv(f"{RAW}/olist_customers_dataset.csv",
                          dtype={"customer_zip_code_prefix": str})
products    = pd.read_csv(f"{RAW}/olist_products_dataset.csv")
sellers     = pd.read_csv(f"{RAW}/olist_sellers_dataset.csv",
                          dtype={"seller_zip_code_prefix": str})
geolocation = pd.read_csv(f"{RAW}/olist_geolocation_dataset.csv",
                          dtype={"geolocation_zip_code_prefix": str})
translation = pd.read_csv(f"{RAW}/product_category_name_translation.csv")

FILENAMES = {
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "customers":   "olist_customers_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv",
}
print("Loaded 9 staged tables (re-runnable; independent of profiling cells).")

Loaded 9 staged tables (re-runnable; independent of profiling cells).


### Parse Timestamps to datetime64

In [234]:
for c in ["order_purchase_timestamp", "order_approved_at",
          "order_delivered_carrier_date", "order_delivered_customer_date",
          "order_estimated_delivery_date"]:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"], errors="coerce")

for c in ["review_creation_date", "review_answer_timestamp"]:
    reviews[c] = pd.to_datetime(reviews[c], errors="coerce")

print("Timestamps parsed. Null milestones stay NaT (semantic, not errors).")

Timestamps parsed. Null milestones stay NaT (semantic, not errors).


### Normalize ZIP prefixes to have a lenght of 5

In [235]:
for df, col in [(customers,   "customer_zip_code_prefix"),
                (sellers,     "seller_zip_code_prefix"),
                (geolocation, "geolocation_zip_code_prefix")]:
    df[col] = df[col].astype(str).str.strip().str.zfill(5)

print("ZIP prefixes normalized to 5-char zero-padded strings.")
print("e.g. customers:", customers["customer_zip_code_prefix"].head(3).tolist(),
      "| geolocation:", geolocation["geolocation_zip_code_prefix"].head(3).tolist())

ZIP prefixes normalized to 5-char zero-padded strings.
e.g. customers: ['14409', '09790', '01151'] | geolocation: ['01037', '01046', '01046']


### Rename misspelled columns

In [236]:
products = products.rename(columns={
    "product_name_lenght":        "product_name_length",
    "product_description_lenght": "product_description_length",
})
print("Renamed product_*_lenght -> *_length.")

Renamed product_*_lenght -> *_length.


### Fix Translation Typos

In [237]:
FIX = {
    "costruction_tools_garden": "construction_tools_garden",
    "costruction_tools_tools":  "construction_tools_tools",
    "home_confort":             "home_comfort",
    "fashio_female_clothing":   "fashion_female_clothing",
    "arts_and_craftmanship":    "arts_and_crafts",
}
translation["product_category_name_english"] = (
    translation["product_category_name_english"].replace(FIX)
)
print("Translation typos fixed:", len(FIX), "entries.")

Translation typos fixed: 5 entries.


### Remove embedded newlines and whitespaces

In [238]:
for c in ["review_comment_title", "review_comment_message"]:
    reviews[c] = (reviews[c].astype("string")
                          .str.replace(r"\s+", " ", regex=True)
                          .str.strip())
print("Review text normalized (multi-line -> single line). Nulls preserved.")

Review text normalized (multi-line -> single line). Nulls preserved.


### Products Table: fill category and dimension gap

In [239]:
products["product_category_name"] = products["product_category_name"].fillna("unknown")
for c in ["product_name_length", "product_description_length", "product_photos_qty"]:
    products[c] = products[c].fillna(0).astype(int)

for c in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    products[c] = products[c].fillna(products[c].median()).round().astype(int)

if "unknown" not in set(translation["product_category_name"]):
    translation = pd.concat([translation, pd.DataFrame(
        [{"product_category_name": "unknown",
          "product_category_name_english": "unknown"}])], ignore_index=True)

print("Products: category gap -> 'unknown'; counts -> 0; weight/dims -> median.")

Products: category gap -> 'unknown'; counts -> 0; weight/dims -> median.


### Referential Integrity: report and drop orphans (a row in a child table whose foreign key value no longer matches any primary key in the parent table)

In [240]:
def drop_orphans(child, child_col, parent, parent_col, label):
    parent_set = set(parent[parent_col].dropna())
    mask = child[child_col].isin(parent_set)
    n = int((~mask).sum())
    print(f"  {'drop' if n else 'ok  '} {n:>6,} orphans  {label}")
    return child[mask].reset_index(drop=True)

orders      = drop_orphans(orders,      "customer_id", customers, "customer_id",
                           "orders.customer_id -> customers.customer_id")
order_items = drop_orphans(order_items, "order_id",    orders,    "order_id",
                           "order_items.order_id -> orders.order_id")
order_items = drop_orphans(order_items, "product_id",  products,  "product_id",
                           "order_items.product_id -> products.product_id")
order_items = drop_orphans(order_items, "seller_id",   sellers,   "seller_id",
                           "order_items.seller_id -> sellers.seller_id")
payments    = drop_orphans(payments,    "order_id",    orders,    "order_id",
                           "payments.order_id -> orders.order_id")
reviews     = drop_orphans(reviews,     "order_id",    orders,    "order_id",
                           "reviews.order_id -> orders.order_id")
print("Referential integrity enforced.")

  ok        0 orphans  orders.customer_id -> customers.customer_id
  ok        0 orphans  order_items.order_id -> orders.order_id
  ok        0 orphans  order_items.product_id -> products.product_id
  ok        0 orphans  order_items.seller_id -> sellers.seller_id
  ok        0 orphans  payments.order_id -> orders.order_id
  ok        0 orphans  reviews.order_id -> orders.order_id
Referential integrity enforced.


### Drop Duplicates 

In [241]:
n_geo = len(geolocation)
geolocation = geolocation.drop_duplicates(
    subset=["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"],
    keep="first").reset_index(drop=True)
print(f"geolocation: {n_geo:,} -> {len(geolocation):,} rows "
      f"({n_geo - len(geolocation):,} duplicate coordinates removed)")

orders      = orders.drop_duplicates().reset_index(drop=True)
order_items = order_items.drop_duplicates().reset_index(drop=True)
payments    = payments.drop_duplicates().reset_index(drop=True)
reviews     = reviews.drop_duplicates().reset_index(drop=True)
customers   = customers.drop_duplicates().reset_index(drop=True)
products    = products.drop_duplicates().reset_index(drop=True)
sellers     = sellers.drop_duplicates().reset_index(drop=True)
translation = translation.drop_duplicates().reset_index(drop=True)
print("Full-row dedupe applied to all tables.")

geolocation: 1,000,163 -> 720,154 rows (280,009 duplicate coordinates removed)
Full-row dedupe applied to all tables.


### Validation by assert statements

In [242]:
print("CLEANED LAYER SUMMARY")
for name, df in [("orders", orders), ("order_items", order_items),
                 ("payments", payments), ("reviews", reviews),
                 ("customers", customers), ("products", products),
                 ("sellers", sellers), ("geolocation", geolocation),
                 ("translation", translation)]:
    print(f"  {name:12s} {len(df):>9,} rows  x  {df.shape[1]:>2} cols")

assert orders["order_status"].isin(
    ["created", "approved", "invoiced", "processing", "shipped",
     "delivered", "unavailable", "canceled"]).all()
assert reviews["review_score"].between(1, 5).all()
assert (products[["product_weight_g", "product_length_cm",
                  "product_height_cm", "product_width_cm"]] >= 0).all().all()
assert (customers["customer_zip_code_prefix"].str.len() == 5).all()
assert (sellers["seller_zip_code_prefix"].str.len() == 5).all()
assert (geolocation["geolocation_zip_code_prefix"].str.len() == 5).all()
print("All post-clean assertions passed.")

os.makedirs("cleaned", exist_ok=True)
for name, df in [("orders", orders), ("order_items", order_items),
                 ("payments", payments), ("reviews", reviews),
                 ("customers", customers), ("products", products),
                 ("sellers", sellers), ("geolocation", geolocation),
                 ("translation", translation)]:
    df.to_csv(f"cleaned/{FILENAMES[name]}", index=False)
print("Saved cleaned CSVs to 'cleaned/'.")

CLEANED LAYER SUMMARY
  orders          99,441 rows  x   8 cols
  order_items    112,650 rows  x   7 cols
  payments       103,886 rows  x   5 cols
  reviews         99,224 rows  x   7 cols
  customers       99,441 rows  x   5 cols
  products        32,951 rows  x   9 cols
  sellers          3,095 rows  x   4 cols
  geolocation    720,154 rows  x   5 cols
  translation         72 rows  x   2 cols
All post-clean assertions passed.
Saved cleaned CSVs to 'cleaned/'.
